# GDELT Historical Sentiment Fetch + FinBERT Scoring (run on Colab)

Fetches dated news articles about a company across a historical window (in weekly chunks, to work around GDELT's per-request result cap), scores each headline with real FinBERT, and produces a CSV.

**How to use:** Runtime menu -> Run all. After the first chunk fetches, check the printed sample titles actually look relevant before letting the rest run (a previous attempt had a broken query that silently returned generic global news instead of company-specific articles). When done, `gdelt_sentiment.csv` downloads automatically.

In [ ]:
!pip install -q transformers --quiet

In [ ]:
import urllib.request, json, urllib.parse, time
from datetime import datetime, timedelta, timezone
import pandas as pd


def fetch_with_retry(url, max_attempts=8, base_delay=10):
    for attempt in range(max_attempts):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            return json.loads(urllib.request.urlopen(req, timeout=25).read())
        except Exception as e:
            print(f"  attempt {attempt+1} failed: {e}", flush=True)
            time.sleep(base_delay)
    return None


# Plain "Google" full-text search matches the word appearing ANYWHERE in an
# article (e.g. "hosted on Google Cloud" mentioned in passing), so most results
# turned out to be unrelated noise. Restricting to known business/tech news
# domains is the standard fix — cuts most of the incidental-mention noise out.
QUERY = '"Google" (domainis:techcrunch.com OR domainis:reuters.com OR domainis:cnbc.com OR domainis:bloomberg.com OR domainis:theverge.com OR domainis:arstechnica.com OR domainis:engadget.com)'
DAYS_BACK = 90  # GDELT DOC 2.0 API officially supports a rolling ~3 month window

query_encoded = urllib.parse.quote(QUERY)
end = datetime.now(timezone.utc).replace(tzinfo=None)
start = end - timedelta(days=DAYS_BACK)

all_articles = []
cursor = start
week = timedelta(days=7)
first_chunk = True
while cursor < end:
    chunk_end = min(cursor + week, end)
    s = cursor.strftime("%Y%m%d%H%M%S")
    e = chunk_end.strftime("%Y%m%d%H%M%S")
    url = (f'https://api.gdeltproject.org/api/v2/doc/doc?query={query_encoded}'
           f'&mode=ArtList&format=json&maxrecords=250&sort=DateDesc'
           f'&startdatetime={s}&enddatetime={e}')
    print(f"fetching {cursor.date()} to {chunk_end.date()}...", flush=True)
    data = fetch_with_retry(url)
    if data:
        arts = data.get('articles', [])
        print(f"  got {len(arts)} articles", flush=True)
        all_articles.extend(arts)
        if first_chunk and arts:
            print("  SAMPLE TITLES (check these are actually Google-related):")
            for a in arts[:5]:
                print("   -", a.get('title'))
            first_chunk = False
    else:
        print("  gave up on this chunk", flush=True)
    cursor = chunk_end
    time.sleep(3)

print(f"\nTOTAL: {len(all_articles)} articles")
df = pd.DataFrame(all_articles)
if not df.empty:
    df['date'] = df['seendate'].str[:8]
    print(f"unique dates: {df['date'].nunique()}, range: {df['date'].min()} to {df['date'].max()}")
df.head()

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, pipeline

# Loaded via the specific Bert classes (not AutoConfig/AutoModel) because
# finbert-tone's config.json predates the model_type field newer transformers
# releases require for Auto* dispatch — this is also the officially documented
# loading method on the model's own Hugging Face page.
tokenizer = BertTokenizer.from_pretrained("yiyanghkust/finbert-tone")
model = BertForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone", num_labels=3)
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)


def signed_score(title: str) -> float:
    result = classifier(title[:512])[0]
    sign = {"Positive": 1, "Negative": -1, "Neutral": 0}.get(result["label"], 0)
    return sign * result["score"]


if not df.empty:
    print(f"Scoring {len(df)} headlines with FinBERT...")
    df["sentiment"] = df["title"].apply(signed_score)
    print("Done. Sample:")
    print(df[["date", "title", "sentiment"]].head())
else:
    print("No articles to score.")

In [ ]:
OUTPUT_FILE = "gdelt_sentiment.csv"
df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(df)} scored headlines to {OUTPUT_FILE}")

try:
    from google.colab import files
    files.download(OUTPUT_FILE)
except ImportError:
    print("Not running on Colab — find the file in the current directory instead.")